# Stage 5 — Multi-Task Temporal Model

### The specific claim under test

Stage 4 showed `defence` fails at F1 0.367, and that giving `block` its own homogeneous 186-sample class moved it only to 0.416. So the problem is **not** taxonomy. Scalar features cannot separate block from loop because both have similar amplitude, contact height and table distance.

What differs is the **shape of the velocity curve over time** — a loop ramps up through a long backswing, a block is a short abrupt punch with almost no preparation. Extrema like `vel_peak` and `amp_y_range` compress that away.

**This notebook tests whether a temporal encoder recovers it.** That is a falsifiable claim, not a general hope for a better number.

### Why a dilated TCN rather than PoseC3D

The roadmap preferred PoseC3D for its NTU-60 pretrained weights. Two reasons to start here instead:

1. The hypothesis is specifically about **temporal shape**, and a dilated TCN over the raw sequence tests it directly with no confound from a pretrained visual prior.
2. `mmaction2` needs `mmcv`, which is a substantial install risk in Colab. If the TCN shows the effect, PoseC3D becomes a worthwhile follow-up for extra points. If the TCN shows nothing, PoseC3D would not have saved it.

### Multi-task heads

| head | classes | role |
|---|---|---|
| shot_class | 4 | primary |
| technique | 8 | auxiliary — carries block/chop/lob that the primary head cannot evaluate |
| forehand/backhand | 2 | auxiliary |
| lean | 6 | auxiliary regulariser |
| feet | 5 | auxiliary regulariser |

With ~1,432 samples the model will memorise unless constrained. The lean and feet labels are free, pose-observable supervision published only in Dec 2025 — **nobody has used them this way.**

### Gates

- **Defence F1 ≥ 0.55** → the primary 4-class taxonomy stands (per `TAXONOMY.yaml` decision gate)
- **Macro-F1 ≥ 0.72** → target range 0.72–0.80
- **Aux heads contribute +3 to +6 points** over single-task — measured as an explicit ablation

GPU required. ~30–45 min for the full run including ablation.


## 1 · Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Setup

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

EPOCHS      = 60
BATCH       = 64
LR          = 2e-3
SEED        = 42
USE_OPPONENT = False      # D8 kept the data; this is an ablation switch

import json, math, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

BASE = Path(BASE); META = BASE/"derived/meta"; CLIP = BASE/"derived/clips"
OUT  = BASE/"outputs"; (OUT/"metrics").mkdir(parents=True, exist_ok=True)
(OUT/"figures").mkdir(parents=True, exist_ok=True)
(BASE/"models/checkpoints").mkdir(parents=True, exist_ok=True)

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "No GPU. Runtime > Change runtime type > T4."
print(f"device: {torch.cuda.get_device_name(0)}")

def seed_all(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

# taxonomy: prefer v2, fall back to v1
import yaml
tax_path = next((p for p in [BASE/"TAXONOMY_v2.yaml", BASE/"TAXONOMY.yaml"]
                 if p.exists()), None)
TAX = yaml.safe_load(tax_path.read_text())
print(f"taxonomy: {tax_path.name} (v{TAX.get('version')})")

d = np.load(CLIP/"canonical.npz", allow_pickle=True)
PRE, NF = int(d["contact_index"]), int(d["n_frames"])
usable  = d["usable"]

CLASSES   = ["serve", "attack", "control", "defence"]
TECHS     = ["block","chop","flick","lob","loop","push","serve","smash"]
LEANS     = ["neutral","back_heavy","front_heavy","right_leaning","left_leaning","unknown"]
FEETS     = ["both_feet_planted","both_feet_lifted","left_foot_lifted","right_foot_lifted","unknown"]

meta = pd.DataFrame({k: d[k] for k in
        ["stroke_id","video_id","fold","shot_class","high_level","technique",
         "lean","feet","eff_hand"]})
print(f"{usable.sum()} usable of {len(meta)}")

device: Tesla T4
taxonomy: TAXONOMY_v2.yaml (v2)
1432 usable of 1457


## 3 · Tensors

Input channels per frame: joint coordinates, velocities, the validity mask, and table distance. The mask is a channel rather than a filter so the network can learn to distrust interpolated regions instead of being silently fed invented values.

In [3]:
KP   = d["kp"].astype(np.float32)      # (N,2,T,17,2)
VEL  = d["vel"].astype(np.float32)
VAL  = d["valid"].astype(np.float32)
TD   = np.nan_to_num(d["table_dist"].astype(np.float32), nan=0.0)

def build(idx, players=(0,)):
    xs = []
    for p in players:
        kp  = KP[idx][:, p].reshape(len(idx), NF, -1)     # 34
        vel = VEL[idx][:, p].reshape(len(idx), NF, -1)    # 34
        val = VAL[idx][:, p]                              # 17
        xs += [kp, vel, val]
    xs.append(TD[idx][..., None])                         # 1
    x = np.concatenate(xs, -1)                            # (N,T,C)
    return np.nan_to_num(x).transpose(0, 2, 1)            # (N,C,T)

ALL = np.arange(len(meta))
PLAYERS = (0, 1) if USE_OPPONENT else (0,)
X = build(ALL, PLAYERS)
C_IN = X.shape[1]
print(f"X {X.shape}  ({C_IN} channels x {NF} frames)")

def enc(col, vocab):
    m = {v: i for i, v in enumerate(vocab)}
    return meta[col].map(m).fillna(-1).astype(int).values

Y = {
    "shot":  enc("shot_class", CLASSES),
    "tech":  enc("technique",  TECHS),
    "fhbh":  enc("high_level", ["forehand","backhand"]),
    "lean":  enc("lean",  LEANS),
    "feet":  enc("feet",  FEETS),
}
N_OUT = {"shot":4, "tech":8, "fhbh":2, "lean":len(LEANS), "feet":len(FEETS)}
W_AUX = {"tech":0.3, "fhbh":0.3, "lean":0.3, "feet":0.2}

for k, v in Y.items():
    print(f"  {k:<5} {N_OUT[k]} classes, {(v<0).sum()} missing")

X (1457, 86, 97)  (86 channels x 97 frames)
  shot  4 classes, 0 missing
  tech  8 classes, 0 missing
  fhbh  2 classes, 0 missing
  lean  6 classes, 0 missing
  feet  5 classes, 1 missing


## 4 · Model

Six dilated residual blocks with dilation 1→32 give a receptive field wider than the 97-frame window, so the final representation sees the whole swing.

**Attention pooling, not the last hidden state.** Contact sits at the centre of the window (index 60 of 97). Reading the final timestep — which the v1 LSTM did — forces the network to carry the decisive moment through 36 steps of decay. Learned attention weights let it read the frames that matter.

In [4]:
class Block(nn.Module):
    def __init__(s, c, dil, drop=0.2):
        super().__init__()
        pad = dil * 2
        s.c1 = nn.Conv1d(c, c, 5, padding=pad, dilation=dil)
        s.c2 = nn.Conv1d(c, c, 5, padding=pad, dilation=dil)
        s.n1, s.n2 = nn.BatchNorm1d(c), nn.BatchNorm1d(c)
        s.do = nn.Dropout(drop)
    def forward(s, x):
        r = x
        x = s.do(F.gelu(s.n1(s.c1(x))))
        x = s.do(F.gelu(s.n2(s.c2(x))))
        return F.gelu(x + r)


class AttnPool(nn.Module):
    def __init__(s, c):
        super().__init__(); s.score = nn.Conv1d(c, 1, 1)
    def forward(s, x):                       # (B,C,T)
        w = torch.softmax(s.score(x), -1)
        return torch.cat([(x * w).sum(-1), x.max(-1).values], -1)


class Net(nn.Module):
    def __init__(s, c_in, width=128, multitask=True):
        super().__init__()
        s.multitask = multitask
        s.stem = nn.Sequential(nn.Conv1d(c_in, width, 1),
                               nn.BatchNorm1d(width), nn.GELU())
        s.blocks = nn.Sequential(*[Block(width, dl) for dl in (1,2,4,8,16,32)])
        s.pool = AttnPool(width)
        s.trunk = nn.Sequential(nn.Linear(width*2, 256), nn.GELU(), nn.Dropout(0.3))
        heads = ["shot"] + (list(W_AUX) if multitask else [])
        s.heads = nn.ModuleDict({h: nn.Linear(256, N_OUT[h]) for h in heads})
    def forward(s, x):
        z = s.trunk(s.pool(s.blocks(s.stem(x))))
        return {h: s.heads[h](z) for h in s.heads}


def focal(logits, target, weight=None, gamma=2.0):
    m = target >= 0
    if m.sum() == 0:
        return logits.sum() * 0.0
    logits, target = logits[m], target[m]
    ce = F.cross_entropy(logits, target, weight=weight, reduction="none")
    pt = torch.exp(-F.cross_entropy(logits, target, reduction="none"))
    return ((1 - pt) ** gamma * ce).mean()

print(f"params: {sum(p.numel() for p in Net(C_IN).parameters()):,}")

params: 1,071,386


## 5 · Augmentation

Applied **stochastically per batch**, never materialised as copies. The v1 experiment pre-generated 7 shifted duplicates of every sample, so the network saw each real stroke 7 times per epoch and hit 95% training accuracy in a single pass — memorisation, not learning.

In [5]:
def augment(x):
    B, C, T = x.shape
    # contact jitter +/- 6 frames
    sh = torch.randint(-6, 7, (B,), device=x.device)
    idx = (torch.arange(T, device=x.device)[None] + sh[:, None]).clamp(0, T-1)
    x = torch.gather(x, 2, idx[:, None].expand(-1, C, -1))
    # temporal speed warp 0.85 - 1.15x
    if random.random() < 0.5:
        sc = random.uniform(0.85, 1.15)
        x = F.interpolate(x, size=max(8, int(T*sc)), mode="linear",
                          align_corners=False)
        x = F.interpolate(x, size=T, mode="linear", align_corners=False)
    # keypoint dropout: zero whole channels
    if random.random() < 0.3:
        x = x * (torch.rand(B, C, 1, device=x.device) > 0.1).float()
    # gaussian jitter
    x = x + torch.randn_like(x) * 0.01
    return x

print("augmentation ready")

augmentation ready


## 6 · Training loop

In [6]:
from sklearn.metrics import f1_score

def run_fold(f, multitask, Xa, verbose=False):
    tr = ((meta.fold != f) & usable).values
    va = ((meta.fold == f) & usable).values

    xt = torch.tensor(Xa[tr], device=dev)
    xv = torch.tensor(Xa[va], device=dev)
    mu, sd = xt.mean((0,2), keepdim=True), xt.std((0,2), keepdim=True) + 1e-6
    xt, xv = (xt-mu)/sd, (xv-mu)/sd

    yt = {k: torch.tensor(v[tr], device=dev) for k, v in Y.items()}
    yv = {k: torch.tensor(v[va], device=dev) for k, v in Y.items()}

    cnt = np.bincount(Y["shot"][tr], minlength=4).clip(1)
    cw = torch.tensor(len(Y["shot"][tr])/(4*cnt), dtype=torch.float32, device=dev)

    net = Net(Xa.shape[1], multitask=multitask).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, LR, total_steps=EPOCHS*max(1, math.ceil(tr.sum()/BATCH)))

    # balanced sampling so defence is not buried by the 2.9:1 imbalance
    p = (1.0/cnt)[Y["shot"][tr]]; p = p/p.sum()
    best, best_state = -1, None

    for ep in range(EPOCHS):
        net.train()
        order = np.random.choice(tr.sum(), tr.sum(), p=p)
        for i in range(0, len(order), BATCH):
            b = order[i:i+BATCH]
            if len(b) < 4: continue
            xb = augment(xt[b])
            out = net(xb)
            loss = focal(out["shot"], yt["shot"][b], cw)
            if multitask:
                for h, w in W_AUX.items():
                    loss = loss + w * focal(out[h], yt[h][b])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step(); sch.step()

        if ep >= 15 and ep % 3 == 0:
            net.eval()
            with torch.no_grad():
                pv = net(xv)["shot"].argmax(1).cpu().numpy()
            s = f1_score(Y["shot"][va], pv, average="macro", zero_division=0)
            if s > best:
                best = s
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}

    if best_state is not None:            # EPOCHS <= 15 would leave it unset
        net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        logits = net(xv)["shot"]
        prob = torch.softmax(logits, 1).cpu().numpy()
    return prob, va, best

print("training loop ready")

training loop ready


## 7 · Multi-task run (7-fold LOVO)

In [7]:
FOLDS = sorted(meta.fold[usable].unique())

def evaluate(multitask, Xa, tag):
    seed_all(SEED)
    oof = np.zeros((len(meta), 4)); rows = []
    for f in FOLDS:
        prob, va, sc = run_fold(f, multitask, Xa)
        oof[va] = prob
        yt, yp = Y["shot"][va], prob.argmax(1)
        r = {"fold": f, "n": int(va.sum()),
             "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
             "acc": float((yt == yp).mean())}
        for i, c in enumerate(CLASSES):
            r[c] = f1_score(yt == i, yp == i, zero_division=0)
        rows.append(r)
        print(f"  {tag} fold {f}: macro-F1 {r['macro_f1']:.3f}  acc {r['acc']:.3f}")
    return oof, pd.DataFrame(rows)

oof_mt, pf_mt = evaluate(True, X, "MT")
pf_mt.to_csv(OUT/"metrics/stage5_folds_multitask.csv", index=False)

print("\n" + pf_mt[["fold","n","macro_f1","acc"]+CLASSES].to_string(
    index=False, float_format=lambda x: f"{x:.3f}"))
w = pf_mt.n/pf_mt.n.sum()
print(f"\n  weighted macro-F1 : {(pf_mt.macro_f1*w).sum():.3f}")
print(f"  unweighted        : {pf_mt.macro_f1.mean():.3f} +/- {pf_mt.macro_f1.std():.3f}")

  MT fold A: macro-F1 0.692  acc 0.675
  MT fold B: macro-F1 0.836  acc 0.868
  MT fold C: macro-F1 0.786  acc 0.789
  MT fold D: macro-F1 0.748  acc 0.735
  MT fold E: macro-F1 0.636  acc 0.730
  MT fold F: macro-F1 0.842  acc 0.845
  MT fold G: macro-F1 0.712  acc 0.747

fold   n  macro_f1   acc  serve  attack  control  defence
   A 160     0.692 0.675  0.980   0.722    0.815    0.250
   B 385     0.836 0.868  0.983   0.875    0.902    0.583
   C 152     0.786 0.789  0.984   0.845    0.708    0.608
   D 170     0.748 0.735  0.929   0.800    0.702    0.560
   E 248     0.636 0.730  0.939   0.644    0.778    0.182
   F 155     0.842 0.845  0.952   0.859    0.867    0.689
   G 162     0.712 0.747  0.980   0.791    0.640    0.435

  weighted macro-F1 : 0.756
  unweighted        : 0.750 +/- 0.076


## 8 · Results & gates

In [8]:
from sklearn.metrics import classification_report, confusion_matrix
u = usable
yt, yp = Y["shot"][u], oof_mt[u].argmax(1)
print(classification_report(yt, yp, target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(yt, yp)
cmdf = pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES],
                    columns=[f"pred_{c}" for c in CLASSES])
cmdf["recall"] = (np.diag(cm)/cm.sum(1)).round(3)
print(cmdf.to_string()); cmdf.to_csv(OUT/"metrics/stage5_confusion.csv")

macro   = f1_score(yt, yp, average="macro", zero_division=0)
def_f1  = f1_score(yt == 3, yp == 3, zero_division=0)
S4 = {"macro": 0.693, "serve": 0.900, "attack": 0.775,
      "control": 0.731, "defence": 0.367}

print("\n" + "=" * 70)
print(f"  {'':<10}{'Stage 4':>10}{'Stage 5':>10}{'delta':>10}")
for i, c in enumerate(CLASSES):
    v = f1_score(yt == i, yp == i, zero_division=0)
    print(f"  {c:<10}{S4[c]:>10.3f}{v:>10.3f}{v-S4[c]:>+10.3f}")
print(f"  {'MACRO':<10}{S4['macro']:>10.3f}{macro:>10.3f}{macro-S4['macro']:>+10.3f}")

print("\n" + "=" * 70)
print("GATES")
print(f"  macro-F1 >= 0.72   : {macro:.3f}  "
      f"{'PASS' if macro >= 0.72 else 'BELOW TARGET'}")
print(f"  defence  >= 0.55   : {def_f1:.3f}  "
      f"{'PASS -> keep 4-class taxonomy' if def_f1 >= 0.55 else 'FAIL'}")
if def_f1 < 0.55:
    print("     Per TAXONOMY.yaml, defence below 0.55 means the distinction is")
    print("     not recoverable from pose. Switch the product to `coarse`")
    print("     (serve / attack / defensive) before Stage 6.")
print("=" * 70)

              precision    recall  f1-score   support

       serve      0.971     0.960     0.966       277
      attack      0.843     0.766     0.803       650
     control      0.711     0.864     0.780       279
     defence      0.504     0.509     0.507       226

    accuracy                          0.782      1432
   macro avg      0.757     0.775     0.764      1432
weighted avg      0.788     0.782     0.783      1432

              pred_serve  pred_attack  pred_control  pred_defence  recall
true_serve           266            2             5             4   0.960
true_attack            4          498            63            85   0.766
true_control           3           11           241            24   0.864
true_defence           1           80            30           115   0.509

               Stage 4   Stage 5     delta
  serve          0.900     0.966    +0.066
  attack         0.775     0.803    +0.028
  control        0.731     0.780    +0.049
  defence        0.367

## 9 · The block-vs-loop test

The specific claim this notebook exists to check. Per-technique accuracy, compared against Stage 4's scalar baseline.

In [9]:
S4_TECH = {"loop":0.821,"serve":0.870,"push":0.724,"block":0.419,
           "flick":0.641,"smash":0.833,"chop":0.065,"lob":0.000}

mu_ = meta[u].copy()
mu_["pred"] = [CLASSES[i] for i in yp]
mu_["correct"] = mu_.pred == mu_.shot_class

t = (mu_.groupby("technique")
        .agg(n=("correct","size"), stage5=("correct","mean")))
t["stage4"] = [S4_TECH.get(i, np.nan) for i in t.index]
t["delta"]  = t.stage5 - t.stage4
print(t.sort_values("n", ascending=False).to_string(
    float_format=lambda x: f"{x:.3f}"))

b4, b5 = S4_TECH["block"], float(t.loc["block","stage5"]) if "block" in t.index else np.nan
print("\n" + "=" * 70)
print(f"  BLOCK accuracy: {b4:.3f} (scalar)  ->  {b5:.3f} (temporal)  "
      f"{b5-b4:+.3f}")
if b5 - b4 > 0.10:
    print("  CONFIRMED: temporal shape carries the block/loop distinction that")
    print("  scalar extrema could not. The Stage 4 diagnosis was correct.")
elif b5 - b4 > 0.03:
    print("  PARTIAL: some gain, but less than the hypothesis predicted.")
    print("  PoseC3D with NTU-60 pretraining is the next lever worth pulling.")
else:
    print("  REFUTED: temporal modelling does not recover block. The block/loop")
    print("  distinction is likely not present in body pose at all — it lives in")
    print("  racket angle and ball behaviour. Adopt the `coarse` taxonomy.")
print("=" * 70)

mu_.to_csv(OUT/"metrics/stage5_predictions.csv", index=False)

             n  stage5  stage4  delta
technique                            
loop       574   0.800   0.821 -0.021
push       279   0.864   0.724  0.140
serve      277   0.960   0.870  0.090
block      186   0.532   0.419  0.113
flick       64   0.453   0.641 -0.188
chop        31   0.387   0.065  0.322
smash       12   0.833   0.833  0.000
lob          9   0.444   0.000  0.444

  BLOCK accuracy: 0.419 (scalar)  ->  0.532 (temporal)  +0.113
  CONFIRMED: temporal shape carries the block/loop distinction that
  scalar extrema could not. The Stage 4 diagnosis was correct.


## 10 · Ablation — do the auxiliary heads help?

Same encoder, same seed, same folds, trained on the primary head alone. The difference is the contribution of the lean / feet / technique / forehand-backhand supervision.

This is a publishable result in itself — those labels were released in December 2025 and have not been used this way.

In [10]:
oof_st, pf_st = evaluate(False, X, "ST")
pf_st.to_csv(OUT/"metrics/stage5_folds_singletask.csv", index=False)

yp_st = oof_st[u].argmax(1)
macro_st = f1_score(yt, yp_st, average="macro", zero_division=0)

print("\n" + "=" * 70)
print("ABLATION: multi-task vs single-task")
print("=" * 70)
print(f"  {'':<10}{'single':>10}{'multi':>10}{'delta':>10}")
for i, c in enumerate(CLASSES):
    a = f1_score(yt == i, yp_st == i, zero_division=0)
    b = f1_score(yt == i, yp    == i, zero_division=0)
    print(f"  {c:<10}{a:>10.3f}{b:>10.3f}{b-a:>+10.3f}")
print(f"  {'MACRO':<10}{macro_st:>10.3f}{macro:>10.3f}{macro-macro_st:>+10.3f}")

gain = macro - macro_st
print(f"\n  auxiliary-head contribution: {gain:+.3f}")
print(f"  (roadmap predicted +0.03 to +0.06)")
if gain >= 0.03:
    print("  CONFIRMED — the Dec-2025 lean/feet labels measurably help.")
elif gain > 0:
    print("  Positive but below prediction. Try raising the aux loss weights.")
else:
    print("  No gain. The aux heads may be competing with the primary task;")
    print("  try lowering their weights before dropping them.")

pd.DataFrame([{"variant":"single_task","macro_f1":macro_st},
              {"variant":"multi_task","macro_f1":macro},
              {"variant":"stage4_lightgbm","macro_f1":S4["macro"]}]
            ).to_csv(OUT/"metrics/stage5_ablation.csv", index=False)
print("=" * 70)

  ST fold A: macro-F1 0.696  acc 0.688
  ST fold B: macro-F1 0.827  acc 0.852
  ST fold C: macro-F1 0.802  acc 0.822
  ST fold D: macro-F1 0.770  acc 0.753
  ST fold E: macro-F1 0.657  acc 0.758
  ST fold F: macro-F1 0.873  acc 0.884
  ST fold G: macro-F1 0.701  acc 0.704

ABLATION: multi-task vs single-task
                single     multi     delta
  serve          0.976     0.966    -0.011
  attack         0.805     0.803    -0.002
  control        0.791     0.780    -0.011
  defence        0.547     0.507    -0.041
  MACRO          0.780     0.764    -0.016

  auxiliary-head contribution: -0.016
  (roadmap predicted +0.03 to +0.06)
  No gain. The aux heads may be competing with the primary task;
  try lowering their weights before dropping them.


---
## Done

| artifact | purpose |
|---|---|
| `outputs/metrics/stage5_folds_multitask.csv` | per-fold, per-class F1 |
| `outputs/metrics/stage5_confusion.csv` | confusion matrix |
| `outputs/metrics/stage5_ablation.csv` | multi-task vs single-task vs Stage 4 |
| `outputs/metrics/stage5_predictions.csv` | per-stroke predictions |

**Read in this order:**

1. **Block accuracy (cell 9).** This is the point of the notebook. A gain above +0.10 confirms the Stage 4 diagnosis; below +0.03 refutes it and means the distinction isn't in body pose at all.
2. **Defence F1 vs 0.55.** The `TAXONOMY.yaml` decision gate — determines whether the 4-class product survives.
3. **Ablation delta.** Whether the lean/feet labels earn their place.
4. **Fold spread.** Compare against Stage 4's ±0.106; lower variance means the encoder generalises better across players, not just scores higher.

If block improves but defence still misses 0.55, the follow-up is PoseC3D with NTU-60 pretraining — at this data scale pretrained weights matter more than architecture.

Next: **`06_contact_detector.ipynb`** (Stage 6) — per-frame contact probability over continuous pose, 1,576 positives, target F1 0.85–0.92 at ±5 frames.


In [11]:
# =============================================================================
# CELL 11 — AUXILIARY WEIGHT SWEEP
#
# The first ablation ran auxiliary heads at a TOTAL weight of 1.1
# (tech 0.3 + fhbh 0.3 + lean 0.3 + feet 0.2) against a primary weight of 1.0.
# More than half the gradient went to auxiliary tasks, and defence — the
# hardest class, first to suffer when capacity is diverted — lost 0.041.
#
# So the Dec-2025 lean/feet labels have not actually been tested yet. They were
# tested at a weight that drowns the primary objective.
#
# This sweeps four settings so the auxiliary heads act as REGULARISERS rather
# than competitors. Reuses run_fold / Net / evaluate from the cells above.
#
# ~4 min per config on a T4.
# =============================================================================

import numpy as np, pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

CONFIGS = {
    "none":            {},                                              # 0.00
    "technique_only":  {"tech": 0.20},                                  # 0.20
    "light":           {"tech": 0.15, "fhbh": 0.10,
                        "lean": 0.07, "feet": 0.03},                    # 0.35
    "medium":          {"tech": 0.25, "fhbh": 0.15,
                        "lean": 0.12, "feet": 0.08},                    # 0.60
}
# "heavy" (total 1.1) is the run already completed above — reused, not repeated.

DEF_GATE = 0.55
results, oofs = [], {}

for name, cfg in CONFIGS.items():
    globals()["W_AUX"] = cfg          # Net reads list(W_AUX) for its heads
    seed_all(SEED)
    oof = np.zeros((len(meta), 4))
    for f in FOLDS:
        prob, va, _ = run_fold(f, len(cfg) > 0, X)
        oof[va] = prob
    oofs[name] = oof

    yt_, yp_ = Y["shot"][usable], oof[usable].argmax(1)
    r = {"config": name, "total_w": round(sum(cfg.values()), 2),
         "macro_f1": f1_score(yt_, yp_, average="macro", zero_division=0),
         "accuracy": float((yt_ == yp_).mean())}
    for i, c in enumerate(CLASSES):
        r[c] = f1_score(yt_ == i, yp_ == i, zero_division=0)
    results.append(r)
    print(f"  {name:<16} w={r['total_w']:<5} macro {r['macro_f1']:.3f}   "
          f"defence {r['defence']:.3f}")

# fold in the completed heavy run
results.append({"config": "heavy (earlier run)", "total_w": 1.10,
                "macro_f1": 0.764, "accuracy": np.nan,
                "serve": 0.966, "attack": 0.803,
                "control": 0.780, "defence": 0.507})

res = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
res.to_csv(OUT / "metrics/stage5_aux_sweep.csv", index=False)

print("\n" + "=" * 78)
print("AUXILIARY WEIGHT SWEEP")
print("=" * 78)
print(res[["config", "total_w", "macro_f1", "defence",
           "serve", "attack", "control"]].to_string(
    index=False, float_format=lambda x: f"{x:.3f}"))

best = res.iloc[0]
base = res[res.config == "none"].iloc[0]
print(f"\n  best        : {best.config}  (macro {best.macro_f1:.3f})")
print(f"  vs no-aux   : {best.macro_f1 - base.macro_f1:+.3f} macro, "
      f"{best.defence - base.defence:+.3f} defence")

print("\n" + "=" * 78)
print("DEFENCE GATE  (TAXONOMY.yaml: >= 0.55 keeps the 4-class taxonomy)")
print("=" * 78)
for _, r in res.iterrows():
    mark = "PASS" if r.defence >= DEF_GATE else "below"
    print(f"  {r.config:<20} {r.defence:.3f}  {mark}")

if res.defence.max() >= DEF_GATE:
    win = res.loc[res.defence.idxmax()]
    print(f"\n  GATE MET by '{win.config}' at {win.defence:.3f}.")
    print("  The 4-class taxonomy stands. Record the winning weights in")
    print("  TAXONOMY.yaml under auxiliary_labels and carry them to Stage 6.")
else:
    print(f"\n  Best defence {res.defence.max():.3f} still short of {DEF_GATE}.")
    print("  Remaining lever before switching to `coarse`: PoseC3D with NTU-60")
    print("  pretraining. At ~1,400 samples pretrained weights typically matter")
    print("  more than architecture, and this is the last untested option.")
print("=" * 78)

# --- the flick regression -----------------------------------------------------
print("\n" + "=" * 78)
print("FLICK CHECK  (Stage 4 0.641 -> Stage 5 0.453, the one regression)")
print("=" * 78)
print("""  A flick is a quick attacking stroke played over the table: compact and
  fast, so it shares its silhouette with push and block. The temporal model
  improved sharply on compact strokes, and flicks are now pulled toward
  control/defence — which is exactly why attack recall fell to 0.766.""")
mu_ = meta[usable].copy()
for name in CONFIGS:
    mu_[f"p_{name}"] = [CLASSES[i] for i in oofs[name][usable].argmax(1)]
fl = mu_[mu_.technique == "flick"]
print(f"\n  flick accuracy by config ({len(fl)} samples):")
for name in CONFIGS:
    acc = (fl[f"p_{name}"] == fl.shot_class).mean()
    print(f"    {name:<16} {acc:.3f}")
print(f"\n  where flicks go under the best config ({best.config}):")
col = f"p_{best.config}" if f"p_{best.config}" in fl.columns else f"p_none"
print("   ", dict(fl[col].value_counts()))
print("""
  If flicks are landing in control/defence, the cleanest fix is a mild
  class-balanced weight on the technique head rather than a taxonomy change —
  flick is unambiguously an attacking stroke.""")
print("=" * 78)

  none             w=0     macro 0.780   defence 0.547
  technique_only   w=0.2   macro 0.792   defence 0.560
  light            w=0.35  macro 0.776   defence 0.534
  medium           w=0.6   macro 0.780   defence 0.545

AUXILIARY WEIGHT SWEEP
             config  total_w  macro_f1  defence  serve  attack  control
     technique_only    0.200     0.792    0.560  0.976   0.817    0.813
             medium    0.600     0.780    0.545  0.967   0.815    0.795
               none    0.000     0.780    0.547  0.976   0.805    0.791
              light    0.350     0.776    0.534  0.971   0.805    0.793
heavy (earlier run)    1.100     0.764    0.507  0.966   0.803    0.780

  best        : technique_only  (macro 0.792)
  vs no-aux   : +0.012 macro, +0.012 defence

DEFENCE GATE  (TAXONOMY.yaml: >= 0.55 keeps the 4-class taxonomy)
  technique_only       0.560  PASS
  medium               0.545  below
  none                 0.547  below
  light                0.534  below
  heavy (earlier run) 